# ClearPath incident report comparison

Train a TF-IDF classifier on synthetic report families and evaluate it on fixed, differently worded reports. Compare with a local zero-shot LLM on the same labels. Run from the repository root or the models folder.

In [ ]:
from pathlib import Path
import sys
if Path.cwd().name == 'models':
    sys.path.insert(0, str(Path.cwd().parent))
from models.benchmark import compare, fit_classifier
from models.reports import TEST_REPORTS, training_reports
print(len(training_reports()), 'training examples')
print(len(TEST_REPORTS), 'test examples')

## Train and inspect a classification

Try an original report and inspect what the bag of words predicts. The model has four output classes.

In [ ]:
classifier = fit_classifier()
example = 'A delivery van broke down across both lanes at Bellevue Hospital.'
print(classifier.predict([example])[0])
print(dict(zip(classifier.classes_, classifier.predict_proba([example])[0].round(3))))

## Evaluate on the fixed test set

Set the local model URL to enable the comparison. Rows of each confusion matrix are true labels, and columns are predictions, including abstentions.

In [ ]:
model_url = None  # use 'http://127.0.0.1:8011' with a running llama.cpp server
results = compare(model_url)
for label in ('tfidf_logistic_regression', 'zero_shot_llm'):
    if label not in results:
        continue
    result = results[label]
    print(label, 'accuracy:', result['accuracy'], 'macro F1:', result['macro_f1'])
    print('true/predicted', *result['labels'], sep='	')
    for name, row in zip(result['labels'], result['matrix']):
        print(name, *row, sep='	')

## Interpret the result

Inspect false positives and abstentions. The reports were authored for this prototype. A student-written held-out set is needed before treating these scores as evidence of field performance.